# Практика: буферы потока заказов

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv():
    for path in (
        Path("orders_slim.csv"),
        Path("../orders_slim.csv"),
        Path("../../data/orders_slim.csv"),
        Path("../data/orders_slim.csv"),
        Path("../../../data/orders_slim.csv"),
    ):
        if path.exists():
            return path.resolve()
    return (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_08_logistics_clustering/data/orders_slim.csv"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")

from collections import deque


## 1. Последние семь событий

Заполните ограниченный deque всем потоком.

In [ ]:
recent = deque(maxlen=7)
# TODO
assert list(recent) == df["order_id"].tail(min(7, len(df))).tolist()


## 2. Скользящее окно

Функция возвращает средние каждого полного окна.

In [ ]:
def rolling_mean(values, width):
    # TODO: deque(maxlen=width)
    ...


means5 = rolling_mean(df["delivery_days"].tolist(), 5)
assert len(means5) == max(0, len(df) - 4)
assert all(isinstance(x, float) for x in means5)


## 3. Две очереди

Разделите поток на late и normal без изменения внутреннего порядка.

In [ ]:
late_q, normal_q = deque(), deque()
# TODO
assert len(late_q) + len(normal_q) == len(df)
assert list(late_q) == df.loc[df["is_late"].eq(1), "order_id"].tolist()


## 4. Три late, затем один normal

Симулируйте цикл до 12 обработок.

In [ ]:
processed = []  # TODO: список пар (order_id, group)
assert len(processed) == min(12, len(df))
assert all(group in {"late", "normal"} for _, group in processed)
assert sum(group == "normal" for _, group in processed[:4]) <= 1


## 5. Остаток очередей

Проверьте закон сохранения числа событий.

In [ ]:
remaining = None  # TODO
assert remaining == len(df) - len(processed)
assert remaining == len(late_q) + len(normal_q)


## 6. Буфер для отмены ошибочной разметки

Последние действия хранятся в stack.

In [ ]:
labels = [(oid, "checked") for oid in df["order_id"].head(6)]
reverted = []  # TODO: отменить два действия через pop
assert [x[0] for x in reverted] == df["order_id"].head(6).tail(2).iloc[::-1].tolist()
assert len(labels) == 4


## 7. Дедупликация соседних событий

Удалите только последовательные повторы, используя последний элемент deque.

In [ ]:
raw = ["A", "A", "B", "B", "A", "C", "C"]
compact = None  # TODO
assert compact == ["A", "B", "A", "C"]


## 8. Эксперимент с квотой

Сравните квоты late 1 и 3 на первых 16 обработках.

In [ ]:
orders_by_quota = {}  # TODO: quota -> список групп
assert set(orders_by_quota) == {1, 3}
assert all(len(v) == min(16, len(df)) for v in orders_by_quota.values())
QUOTA_NOTE = ""  # TODO: >= 100 символов
assert len(QUOTA_NOTE) >= 100


## 9. Самостоятельно: функция процессора

Верните обработанные ids и остатки обеих очередей.

In [ ]:
def process_stream(frame, limit):
    # TODO
    ...


done, late_left, normal_left = process_stream(df, min(15, len(df)))
assert len(done) == min(15, len(df))
assert len(done) + late_left + normal_left == len(df)
assert len(done) == len(set(done))
